# Math 16 · Synthesis and Derivation Workshop

This notebook is the math gate before the final capstone.

Do not merely run it. For each problem:
1. derive the result on paper or in a Markdown cell;
2. predict the numerical result;
3. run the verification cell;
4. explain any discrepancy.

## Problem 1 · Least-squares gradient

Given

$$
L(w)=\frac1n\|Xw-y\|^2,
$$

derive $\nabla_wL$ and state its shape.

In [ ]:
import numpy as np
rng=np.random.default_rng(0)
X=rng.normal(size=(20,4)); y=rng.normal(size=20); w=rng.normal(size=4)
analytic=2/len(X)*X.T@(X@w-y)
eps=1e-6
numeric=np.array([
 ((np.mean((X@(w+eps*np.eye(4)[j])-y)**2))-(np.mean((X@(w-eps*np.eye(4)[j])-y)**2)))/(2*eps)
 for j in range(4)
])
print("gradient check:",np.allclose(analytic,numeric,rtol=1e-5,atol=1e-5))
print("shape:",analytic.shape)

## Problem 2 · Bayes and base rates

A detector has $P_D=0.9$, $P_{FA}=0.01$, and signal prevalence $P(H_1)=10^{-3}$.

Derive $P(H_1\mid +)$. Predict whether it is closer to 0.1, 0.5, or 0.9 before running.

In [ ]:
pd=.9; pfa=.01; prior=1e-3
post=pd*prior/(pd*prior+pfa*(1-prior))
print(post)

## Problem 3 · Cross-entropy gradient

For softmax probabilities $p$ and one-hot target $y$, derive

$$
\frac{\partial L}{\partial z}=p-y.
$$

Then verify with finite differences.

In [ ]:
def softmax(z):
    z=z-z.max(); e=np.exp(z); return e/e.sum()
z=np.array([.2,-.4,1.1]); target=2
p=softmax(z); analytic=p.copy(); analytic[target]-=1
def loss(z): return -np.log(softmax(z)[target])
numeric=np.array([(loss(z+eps*np.eye(3)[j])-loss(z-eps*np.eye(3)[j]))/(2*eps) for j in range(3)])
print("analytic",analytic,"numeric",numeric)

## Problem 4 · PCA

Show that maximizing projected variance

$$
\max_{\|v\|=1} v^TSv
$$

leads to the eigenvector equation $Sv=\lambda v$ using a Lagrange multiplier.

In [ ]:
A=rng.normal(size=(300,3)); A[:,1]=2*A[:,0]+.2*rng.normal(size=300)
S=np.cov(A,rowvar=False)
vals,vecs=np.linalg.eigh(S)
v=vecs[:,-1]
print("top eigenvalue",vals[-1],"projected variance",np.var((A-A.mean(0))@v,ddof=1))

## Problem 5 · Backpropagation shapes

For a network

$$
X_{n\times d}\to W_1{}_{d\times h}\to A_{n\times h}\to W_2{}_{h\times k},
$$

derive the shapes of $\partial L/\partial W_1$ and $\partial L/\partial W_2$.

In [ ]:
n,d,h,k=8,5,7,3
X=rng.normal(size=(n,d)); W1=rng.normal(size=(d,h)); W2=rng.normal(size=(h,k))
Z=X@W1; A=np.maximum(0,Z); dOut=rng.normal(size=(n,k))
dW2=A.T@dOut
dA=dOut@W2.T
dW1=X.T@(dA*(Z>0))
print("dW1",dW1.shape,"dW2",dW2.shape)

## Problem 6 · Attention scaling

Assuming query/key components have variance 1, derive the variance of $q^Tk$ and explain why dividing by $\sqrt{d_k}$ stabilizes it.

In [ ]:
for d in [8,32,128,512]:
    q=rng.normal(size=(10000,d)); k=rng.normal(size=(10000,d))
    dot=(q*k).sum(1)
    print(d,"raw var",dot.var(),"scaled var",(dot/np.sqrt(d)).var())

## Problem 7 · VAE KL

Derive the KL divergence between $\mathcal{N}(\mu,\sigma^2)$ and $\mathcal{N}(0,1)$:

$$
D_{KL}
=
\frac12(\mu^2+\sigma^2-1-\log\sigma^2).
$$

In [ ]:
mu=1.2; sigma=.7
kl=.5*(mu**2+sigma**2-1-np.log(sigma**2))
print("KL",kl)

## Problem 8 · Bellman fixed point

For a deterministic chain with reward 1 at the terminal transition and discount $\gamma$, derive the value at each state by working backward.

In [ ]:
gamma=.9; n=5
V=np.zeros(n)
for s in reversed(range(n-1)):
    V[s]=(1 if s==n-2 else 0)+gamma*V[s+1]
print(V)

## Problem 9 · Matched filter

For $H_0:x=n$ and $H_1:x=s+n$ with $n\sim\mathcal N(0,\sigma^2I)$, derive the log-likelihood ratio and show that the data-dependent term is proportional to $s^Tx$.

In [ ]:
s=rng.normal(size=32); s/=np.linalg.norm(s)
H0=rng.normal(size=(50000,32))
H1=H0[:5000]+1.5*s
score0=H0@s; score1=H1@s
eta=np.quantile(score0,.99)
print("Pfa",np.mean(score0>eta),"Pd",np.mean(score1>eta))

## Final oral defense

Without notes, explain in under ten minutes:

- why gradients have the shapes they do;
- why likelihood connects probability models to losses;
- why regularization changes conditioning/generalization;
- why softmax cross-entropy has the gradient it does;
- why attention scaling is dimension-dependent;
- why Bellman equations are fixed-point equations;
- why the matched filter is optimal only under specific noise/signal assumptions.